In [ ]:
!pip install -q transformers torch torchvision pillow gradio timm ultralytics opencv-python

In [ ]:

from transformers import OwlViTProcessor, OwlViTForObjectDetection
from PIL import Image, ImageDraw
import torch
import gradio as gr
import numpy as np

processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

stored_boxes = []

labels = [
    "lamp",
    "chair",
    "table",
    "sofa",
    "bottle",
    "cup",
    "laptop",
    "phone",
    "book",
    "person",
    "tv",
    "plant",
    "fan",
    "bed",
    "clock"
]

def detect_objects(image):

    global stored_boxes

    stored_boxes = []

    image = image.convert("RGB")

    inputs = processor(
        text=[labels],
        images=image,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.Tensor([image.size[::-1]])

    results = processor.post_process_object_detection(
        outputs=outputs,
        target_sizes=target_sizes,
        threshold=0.15
    )

    result = results[0]

    draw = ImageDraw.Draw(image)

    boxes = result["boxes"]
    scores = result["scores"]
    detected_labels = result["labels"]

    for box, score, label_idx in zip(boxes, scores, detected_labels):

        box = [round(i, 2) for i in box.tolist()]

        x1, y1, x2, y2 = map(int, box)

        label = labels[label_idx]

        stored_boxes.append({
            "label": label,
            "score": float(score),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2
        })

        draw.rectangle(
            [(x1, y1), (x2, y2)],
            outline="red",
            width=3
        )

        draw.text(
            (x1, y1 - 15),
            f"{label} {score:.2f}",
            fill="red"
        )

    return image, stored_boxes

def hotspot_click(evt: gr.SelectData):

    x = evt.index[0]
    y = evt.index[1]

    for item in stored_boxes:

        if (
            item["x1"] <= x <= item["x2"] and
            item["y1"] <= y <= item["y2"]
        ):

            return f"""
Object Found

Name: {item['label']}
Confidence: {item['score']:.2f}

Coordinates:
({item['x1']}, {item['y1']})
({item['x2']}, {item['y2']})
"""

    return "No object detected"

with gr.Blocks() as demo:

    gr.Markdown("# Free AI Image Hotspot Detector")

    gr.Markdown(
        "Upload image → Detect objects → Click object to identify"
    )

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                type="pil",
                label="Upload Image"
            )

            detect_btn = gr.Button("Detect Objects")

        with gr.Column():

            image_output = gr.Image(
                type="pil",
                label="Detected Objects"
            )

    result_box = gr.Textbox(
        label="Hotspot Result",
        lines=8
    )

    json_output = gr.JSON(
        label="Detected Objects JSON"
    )

    detect_btn.click(
        fn=detect_objects,
        inputs=image_input,
        outputs=[image_output, json_output]
    )

    image_output.select(
        fn=hotspot_click,
        outputs=result_box
    )

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

The image processor of type `OwlViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/613M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

OwlViTForObjectDetection LOAD REPORT from: google/owlvit-base-patch32
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
owlvit.text_model.embeddings.position_ids   | UNEXPECTED |  | 
owlvit.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b30aa80ce40a8301c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
